In [ ]:
!pip uninstall websocket
# !pip install websocket-client==0.37.0

In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

ImportError: cannot import name 'WebSocketApp' from 'websocket' (unknown location)

In [4]:
# !pip install --upgrade selenium
# !pip install websocket
# !pip install --upgrade pip

In [ ]:
# Setup Selenium
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

url = "https://www.nseindia.com/report-detail/fo_eq_security"
driver.get(url)

wait = WebDriverWait(driver, 15)

# Example: Selecting Instrument
instrument_dropdown = wait.until(EC.presence_of_element_located((By.ID, "instrumentDropDownId")))
Select(instrument_dropdown).select_by_visible_text("Index Options")

# Example: Selecting Symbol
symbol_dropdown = driver.find_element(By.ID, "symbolDropDownId")
Select(symbol_dropdown).select_by_visible_text("NIFTY")

# Example: Year loop
year_dropdown = driver.find_element(By.ID, "yearDropDownId")
for year in ["2020", "2021", "2022", "2023", "2024", "2025"]:
    Select(year_dropdown).select_by_visible_text(year)

    # Expiry list (pick top 10 dynamically each time)
    expiry_dropdown = driver.find_element(By.ID, "expiryDateDropDownId")
    expiry_options = expiry_dropdown.find_elements(By.TAG_NAME, "option")
    top_10_expiries = [opt.text for opt in expiry_options[:10]]

    for expiry in top_10_expiries:
        Select(expiry_dropdown).select_by_visible_text(expiry)

        # Option type CE and PE
        for option_type in ["CE", "PE"]:
            option_dropdown = driver.find_element(By.ID, "optionTypeDropDownId")
            Select(option_dropdown).select_by_visible_text(option_type)

            # TODO: Select custom date range here (from & to date)

            # Submit the form (button click)
            search_btn = driver.find_element(By.ID, "submitButtonId")
            search_btn.click()

            # Wait for table
            time.sleep(3)

            # Extract table
            dfs = pd.read_html(driver.page_source)
            if dfs:
                df = dfs[0]
                df["Year"] = year
                df["Expiry"] = expiry
                df["OptionType"] = option_type
                print(df.head())

            time.sleep(2)

driver.quit()
